<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/example_op_collections/bundled_ops_hub_optin.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

In [ ]:
# Uncomment to run installation steps if you do not have a development
# editable install and want to run this notebook in a fresh environment.
# %pip install uv
# %uv pip install --upgrade pip setuptools wheel && \
# %uv pip install 'git+https://github.com/speediedan/interpretune.git@main[examples]'
# %uv pip install --group git-deps
#
# NOTE: This cell is intentionally commented out. We will uncomment these
# install commands once we no longer need to preserve editable installs
# for active developer venvs.


# Opting into hub op collections over interpretune's bundled ops

## How op names resolve, before and after an opt-in

Interpretune resolves an op name in one of two ways, and knowing which is which is the whole point of
this notebook.

| You call | Default (no opt-in) | After `it.hub.prefer_ops("org/repo")` |
| --- | --- | --- |
| a **bare** name (`concept_direction`) | the **bundled** op ships with interpretune | that collection's op |
| a **fully-qualified** name (`speediedan.concept_direction_ops.concept_direction`) | that exact op | that exact op (unchanged) |

Three consequences worth holding onto:

- **Bundled ops win bare names by default.** A session behaves the same offline as online, and the same
  for you as for a collaborator who has pulled nothing.
- **Pulling a collection changes nothing on its own.** Fetching adds namespaced ops; it never takes over a
  bare name, so a pull cannot silently re-point code that already works.
- **Fully-qualified names ignore precedence entirely, in both directions.** Explicit beats implicit, which
  is how you pin one specific copy in code that must keep working whatever a session declares.

Opting in is therefore explicit, per-namespace, and reversible within the session. Everything below
demonstrates exactly those rules:

## The arc

1. A session works entirely on bundled ops, with no hub access at all
2. `it.hub.op_info` reports which collection an op name resolves to and at what version
3. Opt into executing hub-resident code (the trust gate)
4. Pull the `concept_direction_ops` collection (manifest-first, revision-pinned)
5. `it.hub.prefer_ops` flips the bare name to the hub copy
6. Fully-qualified names address exactly what they name, regardless of precedence
7. Removing the precedence restores bundled resolution
8. `IT_OP_PRECEDENCE` is the same opt-in for scripted and CLI runs

This notebook only **pulls**; it publishes nothing, so nothing here can modify a repository.

## Setup


In [ ]:
import os

import interpretune as it
import interpretune.analysis  # registers the op wrappers and the dispatcher
from interpretune.analysis import IT_ANALYSIS_CACHE, IT_ANALYSIS_HUB_CACHE

print(f"interpretune {it.__version__}")
print(f"analysis cache:  {IT_ANALYSIS_CACHE}")
print(f"ops hub cache:   {IT_ANALYSIS_HUB_CACHE}")

## Step 1: a session on bundled ops only

No trust opt-in, no network, no hub cache consulted for code. `concept_direction` resolves to the copy that
shipped inside the installed package: the `concept` collection at the version interpretune declares for it.


In [ ]:
info = it.hub.op_info("concept_direction")
print(info)

print()
print(f"provenance:        {info.active.source}")
print(f"collection:        {info.active.collection} {info.active.version}")
print(f"cached revision:   {info.active.revision}  # bundled ops have none -- they are not fetched")

## Step 2: opt into executing hub-resident op code

Loading an op collection from the Hub **executes code published by that repo** in this kernel, so
interpretune refuses unless you opt in. The collection pulled below is generated from interpretune's own
bundled `concept` family, so opting in here is a decision about first-party code; for someone else's repo,
pull it first and read the ops before setting this.

See the [trust posture guide](https://interpretune.org/en/latest/usage/hub_trust_posture.html) for the threat
model and the escape hatches (inspect before trusting, pin a revision, or run with `IT_TRUST_REMOTE_CODE=0`
and no hub code at all).


In [ ]:
os.environ["IT_TRUST_REMOTE_CODE"] = "1"  # default is deny

## Step 3: pull the collection

`it.hub.pull_ops` is manifest-first: it fetches `it_component.yaml`, reads which YAMLs that collection
declares as op definitions, and fetches them **at the commit the manifest resolved to**. A repo updated
mid-resolution therefore cannot hand back op definitions from two different revisions.

**Fetching is the only step that needs the network.** Once a collection is in the ops cache it is
*hub-sourced* but resolved locally: discovery scans the cache, so later sessions load and run those ops
**offline**, at the revision you pulled. The trust gate still applies each session (it governs *executing*
code that came from a repo you do not control, wherever that code now sits on disk), and `op_info` reports
the cached revision without touching the network.

`speediedan/concept_direction_ops` is private until interpretune's Hub library registration lands, so this
cell resolves a token. Once the repo is public the token is unnecessary and the pull works unauthenticated.

In [ ]:
repo_id = "speediedan/concept_direction_ops"
token = os.environ.get("IT_HF_TOKEN") or os.environ.get("HF_TOKEN")

op_files, commit = it.hub.pull_ops(repo_id, token=token)
print(f"pulled {repo_id} at {commit}")
for path in op_files:
    print(f"  op definitions: {path.name}")

# `pull_ops` reloads the dispatcher by default, so the collection is usable immediately. Without that a
# session that had already loaded its ops would not see the new ones and calling one would raise
# `Unknown operation`, with the files sitting on disk the whole time.

## Step 4: both copies are now present, and bundled still wins

Pulling a collection does **not** change what existing code resolves to. That is the point of the default:
a pull cannot silently re-point a colleague's notebook.


In [ ]:
info = it.hub.op_info("concept_direction")
print(info)

print()
print(f"still resolving to bundled: {info.active.source == 'bundled'}")
print(f"shadowing bundled:          {info.is_shadowing_bundled}")

## Step 5: opt into the hub collection's bare names

`prefer_ops` records a namespace whose ops win bare-name resolution. It is per-namespace and explicit: it
affects exactly the op names that collection defines, and nothing else in the session moves.


In [ ]:
print(f"precedence: {it.hub.prefer_ops(repo_id)}")
print()
info = it.hub.op_info("concept_direction")
print(info)
print()
print(f"shadowing bundled: {info.is_shadowing_bundled}")

# A bare-name call such as `it.concept_direction(...)` resolves at dispatch time, so it now reaches the
# hub copy without any call site changing:
print(f"a bare `concept_direction` call now reaches: {info.resolved}")

## Step 6: fully-qualified names ignore precedence entirely

Explicit beats implicit, in both directions: a fully-qualified name always addresses exactly what it names,
whether or not its collection is preferred. This is how you pin one specific copy in code that has to keep
working regardless of what precedence a session declares.

A fully-qualified op name is `<org>.<repo>.<op>` — the repo id with `/` written as `.`:

```python
speediedan.concept_direction_ops.concept_direction    # this collection's op
concept_direction                                     # the bare name, subject to precedence
```

The cell below builds that same string from `repo_id` so the notebook stays copy-pasteable for any
collection, but the literal form above is what you would write in your own code.

In [ ]:
from interpretune.analysis.ops.dispatcher import DISPATCHER

# The fully-qualified name, written out: 'speediedan.concept_direction_ops.concept_direction'.
# Built from repo_id here only so this cell works for any collection.
fq_name = f"{repo_id.replace('/', '.')}.concept_direction"
print(f"fully-qualified name: {fq_name}")

# addressed explicitly, it resolves to itself whether or not the collection is preferred
print(f"resolves to:          {DISPATCHER._resolve_name_safe(fq_name)}")

# every alternative, and which one is active, is inspectable at any point
for candidate in it.hub.op_info("concept_direction").alternatives:
    print(f"  alternative: {candidate}")

## Step 7: remove the precedence and the default returns

Calling `prefer_ops` with no arguments clears the in-process opt-in. Nothing was mutated to make the flip
happen, so nothing has to be repaired to undo it: precedence is applied when a name is resolved, not by
rewriting the op registry.


In [ ]:
print(f"precedence: {it.hub.prefer_ops()}")
print()
print(it.hub.op_info("concept_direction"))

## Step 8: the same opt-in for scripted and CLI runs

`IT_OP_PRECEDENCE` is a comma-separated, ordered list of namespaces, read every time precedence is
consulted. It is the parity surface for runs with no place to call `prefer_ops`:

```bash
IT_OP_PRECEDENCE="speediedan/concept_direction_ops" python -m interpretune ...
```


In [ ]:
os.environ["IT_OP_PRECEDENCE"] = repo_id
print(f"precedence from the environment: {DISPATCHER.op_precedence}")
print(f"resolves to: {it.hub.op_info('concept_direction').resolved}")

del os.environ["IT_OP_PRECEDENCE"]
print()
print(f"precedence after unsetting: {DISPATCHER.op_precedence}")

## Cleanup

Remove the pulled collection from the ops cache so this notebook leaves the environment as it found it. The
reload afterwards is what makes the removal take effect in this kernel.


In [ ]:
import shutil
from pathlib import Path

repo_cache = Path(IT_ANALYSIS_HUB_CACHE) / f"models--{repo_id.replace('/', '--')}"
if repo_cache.exists():
    shutil.rmtree(repo_cache)
    print(f"removed {repo_cache}")

DISPATCHER.reload_definitions()
info = it.hub.op_info("concept_direction")
print(f"\nresolves to: {info.active}")
print(f"alternatives: {list(info.alternatives)}")
assert info.active.source == "bundled" and not info.alternatives, "expected a bundled-only session again"
print("\n\u2713 bundled-only session restored")